In [2]:
# ── Setup ─────────────────────────────────────────────────────────────
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# ── Settings ──────────────────────────────────────────────────────────
AVG_ORDER_VALUE = 120.0  # assumption: source data has no revenue column
CSV_NAME = "digital_marketing_campaign_dataset.csv"

# Folders where the Week 1 CSV might be (checked in this order)
SEARCH_DIRS = [
    Path("."),
    Path("../Week_1"),
    Path("../Week1"),
    Path("../week1"),
    Path("../Week_1/week1"),
    Path("week1"),
    Path(".."),
]


def find_csv(name, search_dirs):
    """Return the first existing path of `name` inside `search_dirs`."""
    for folder in search_dirs:
        candidate = folder / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find '{name}'.\n"
        f"Current folder: {Path.cwd()}\n"
        f"Searched in: {[str(d) for d in search_dirs]}\n"
        "Copy the CSV next to this notebook, or set CSV_PATH manually."
    )


# ── Load data ─────────────────────────────────────────────────────────
CSV_PATH = find_csv(CSV_NAME, SEARCH_DIRS)
customers = pd.read_csv(CSV_PATH)

print(f"Using CSV : {CSV_PATH}")
print(f"Shape     : {customers.shape[0]:,} rows x {customers.shape[1]} columns")

Using CSV : ..\Week1\digital_marketing_campaign_dataset.csv
Shape     : 8,000 rows x 20 columns


In [3]:
# ── Simulated touchpoint log ──────────────────────────────────────────
# The dataset has one row per customer (no real journeys), so we simulate
# a touchpoint-level event log. Same logic as Week 2.

CHANNEL_POOL = ["Social Media", "Email", "PPC", "SEO", "Referral"]
BASE_DATE = datetime(2025, 6, 1)
MAX_TOUCHES = 6
SEED = 42


def generate_touchpoints(customers, seed=SEED):
    """Create 1-6 touchpoints per customer.

    - More engagement (visits + email opens/clicks + shares) -> more touches.
    - The last touch is always the customer's real CampaignChannel and
      carries their real Conversion value.
    - Earlier touches get random channels and earlier timestamps.
    """
    rng = np.random.default_rng(seed)
    records = []

    for row in customers.itertuples(index=False):
        engagement = row.WebsiteVisits + row.EmailOpens + row.EmailClicks + row.SocialShares
        n_touches = int(np.clip(1 + engagement / 25, 1, MAX_TOUCHES))

        # Timestamps: the final touch, plus earlier ones up to 20 days before it
        final_touch = BASE_DATE + timedelta(days=int(rng.integers(0, 25)))
        hours_before = sorted(rng.integers(1, 20 * 24, size=n_touches - 1))
        timestamps = [final_touch - timedelta(hours=int(h)) for h in hours_before] + [final_touch]

        # Channels: random for earlier touches, real channel for the last one
        channels = list(rng.choice(CHANNEL_POOL, size=n_touches - 1)) + [row.CampaignChannel]

        for position, (ts, channel) in enumerate(zip(timestamps, channels), start=1):
            records.append({
                "user_id": row.CustomerID,
                "channel": channel,
                "event_timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
                "is_conversion": int(row.Conversion) if position == n_touches else 0,
            })

    return pd.DataFrame(records)


touchpoints = generate_touchpoints(customers)
print(f"{len(touchpoints):,} touchpoints for {touchpoints['user_id'].nunique():,} customers")

32,268 touchpoints for 8,000 customers


In [4]:
# ── Database + attribution views ──────────────────────────────────────
from pathlib import Path

DB_DIR = Path("data")
DB_PATH = DB_DIR / "attribution.db"
DB_DIR.mkdir(exist_ok=True)

# Step 1: one row per touchpoint of every CONVERTED customer, with its position
JOURNEY_VIEW_SQL = """
SELECT
    t.user_id,
    t.channel,
    ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.event_timestamp)      AS touch_from_start,
    ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.event_timestamp DESC) AS touch_from_end,
    COUNT(*)     OVER (PARTITION BY t.user_id)                                 AS total_touches
FROM touchpoints AS t
WHERE t.user_id IN (SELECT user_id FROM touchpoints WHERE is_conversion = 1)
"""

# Step 2: how much conversion credit each channel gets under each model
CREDIT_VIEWS = {
    # 100% credit to the first touch
    "channel_credit_first_touch": """
        SELECT channel, COUNT(*) * 1.0 AS attributed_conversions
        FROM journey
        WHERE touch_from_start = 1
        GROUP BY channel
    """,
    # 100% credit to the last touch
    "channel_credit_last_touch": """
        SELECT channel, COUNT(*) * 1.0 AS attributed_conversions
        FROM journey
        WHERE touch_from_end = 1
        GROUP BY channel
    """,
    # Credit split equally: 1/n for each of the n touches
    "channel_credit_linear": """
        SELECT channel, SUM(1.0 / total_touches) AS attributed_conversions
        FROM journey
        GROUP BY channel
    """,
}


def create_view(conn, name, select_sql):
    """(Re)create a view so the cell can be re-run safely."""
    conn.execute(f"DROP VIEW IF EXISTS {name}")
    conn.execute(f"CREATE VIEW {name} AS {select_sql}")


conn = sqlite3.connect(DB_PATH)

# Tables
customers.to_sql("customers", conn, index=False, if_exists="replace")
touchpoints.to_sql("touchpoints", conn, index=False, if_exists="replace")

# Views
create_view(conn, "journey", JOURNEY_VIEW_SQL)
for view_name, select_sql in CREDIT_VIEWS.items():
    create_view(conn, view_name, select_sql)

conn.commit()
print("Database ready:", DB_PATH.resolve())

Database ready: C:\Users\Lenovo\infotact solutions\Project 1\Week_3\data\attribution.db


In [5]:
# ── KPIs + star schema ────────────────────────────────────────────────
# Dimension table : dim_channel_spend  -> spend and clicks per channel
# Fact table      : fact_channel_kpis  -> one row per channel x attribution model

ATTRIBUTION_VIEWS = {
    "first_touch": "channel_credit_first_touch",
    "last_touch":  "channel_credit_last_touch",
    "linear":      "channel_credit_linear",
}


def safe_divide(numerator, denominator):
    """Divide, returning NaN instead of an error when the denominator is 0."""
    return numerator / denominator.replace(0, np.nan)


def build_dim_channel(conn):
    """Total ad spend and website visits (used as clicks) for each channel."""
    query = """
        SELECT CampaignChannel AS channel,
               SUM(AdSpend)       AS total_spend,
               SUM(WebsiteVisits) AS total_clicks
        FROM customers
        GROUP BY CampaignChannel
    """
    dim_channel = pd.read_sql(query, conn)
    dim_channel.to_sql("dim_channel_spend", conn, if_exists="replace", index=False)
    return dim_channel


def build_fact_kpis(conn, dim_channel):
    """CPC, CAC, revenue and ROAS per channel for every attribution model."""
    frames = []
    for model_name, view_name in ATTRIBUTION_VIEWS.items():
        credit = pd.read_sql(f"SELECT * FROM {view_name}", conn)

        kpis = credit.merge(dim_channel, on="channel", how="left")
        kpis.insert(0, "attribution_model", model_name)

        kpis["revenue"] = kpis["attributed_conversions"] * AVG_ORDER_VALUE
        kpis["cpc"]  = safe_divide(kpis["total_spend"], kpis["total_clicks"])
        kpis["cac"]  = safe_divide(kpis["total_spend"], kpis["attributed_conversions"])
        kpis["roas"] = safe_divide(kpis["revenue"], kpis["total_spend"])

        frames.append(kpis)

    fact = pd.concat(frames, ignore_index=True)
    fact.to_sql("fact_channel_kpis", conn, if_exists="replace", index=False)
    return fact


# ── Run ───────────────────────────────────────────────────────────────
dim_channel = build_dim_channel(conn)
fact = build_fact_kpis(conn, dim_channel)
conn.commit()

# ── Show results ──────────────────────────────────────────────────────
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Detailed KPIs
fact[["attribution_model", "channel", "total_spend",
      "attributed_conversions", "cac", "roas"]] \
    .sort_values(["attribution_model", "roas"], ascending=[True, False])

,attribution_model,channel,total_spend,attributed_conversions,cac,roas
4,first_touch,Social Media,"7,542,323.2492","1,465.0000","5,148.3435",0.0233
0,first_touch,Email,"7,871,575.8519","1,404.0000","5,606.5355",0.0214
3,first_touch,SEO,"7,740,903.8757","1,377.0000","5,621.5714",0.0213
1,first_touch,PPC,"8,199,236.9811","1,393.0000","5,886.0280",0.0204
2,first_touch,Referral,"8,653,518.6856","1,373.0000","6,302.6356",0.0190
6,last_touch,PPC,"8,199,236.9811","1,461.0000","5,612.0719",0.0214
8,last_touch,SEO,"7,740,903.8757","1,359.0000","5,696.0293",0.0211
7,last_touch,Referral,"8,653,518.6856","1,518.0000","5,700.6052",0.0211
9,last_touch,Social Media,"7,542,323.2492","1,319.0000","5,718.2132",0.0210
5,last_touch,Email,"7,871,575.8519","1,355.0000","5,809.2811",0.0207


In [6]:
fact.pivot(index="channel", columns="attribution_model", values="roas").round(4)

attribution_model,first_touch,last_touch,linear
channel,,,
Email,0.0214,0.0207,0.0209
PPC,0.0204,0.0214,0.0207
Referral,0.0190,0.0211,0.0197
SEO,0.0213,0.0211,0.0216
Social Media,0.0233,0.0210,0.0223


In [7]:
conn.close()